In [ ]:
import os 
import base64
from openai import AzureOpenAI
import json
import re
import numpy as np
import datetime
import pandas as pd
import time
import ast
from collections import defaultdict

In [ ]:
endpoint = "<your_azure_endpoint_here>"
model_name = "gpt-4o"
deployment = "gpt-4o"

subscription_key = "<your_subscription_key_here>"
api_version = "<your_api_version_here>"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

In [ ]:
df_note = pd.read_csv("note_data.csv", encoding='utf-8')
df_note.reset_index(drop=True, inplace=True)

In [ ]:

all_outputs = []

for i, note in enumerate(df_note["note_text"]):
    system_message = """You are a medical expert. Your task is to analyze a medical note and extract referral information in a structured JSON format. Do not include any explanations, disclaimers, or text outside of the JSON object."""
    
    prompt = f"""Follow these steps carefully:
    
    Step 1 – Determine Specialist Referral
    Check if the medical note indicates that the patient was referred to any of the following specialists:
    
    Oncologist, Cardiologist, Gynecologist, Electrophysiologist (cardiology), Lipids Specialist (cardiology), Surgeon (for oncology-related findings), Nephrologist, Endocrinologist, Urologist, Dietician, Nutritionist, Genetic Counselor.
    
    - If a referral to any of these specialists is found, extract and quote the relevant portion of the note and place it in the `referral_confirmation_citation` field.  
    - If no referral to any of the listed specialists is mentioned, set `referral_confirmation_citation` to `null`.
    
    Step 2 – Identify Specialist Type
    If a referral is confirmed, identify the type of specialist (limited to the list above).
    - Quote the part of the note that names the specialist type and place it in `specialist_type_citation`.  
    - If no specific specialist type from the list is provided, set both `specialist_type` and `specialist_type_citation` to `null`.
    
    Step 3 – Output Format
    Return the output in the following JSON structure:
    
    {{
      "referred": true or false,
      "specialists": [
        {{
          "specialist_type": "type_of_specialist" or null,
          "referral_confirmation_citation": "Text from the medical note confirming referral" or null,
          "specialist_type_citation": "Text from the medical note specifying the specialist type" or null
        }}
        ...
        /* repeat for additional specialists if applicable */
      ]
    }}
    
    Now analyze the following medical note: {note}
    
    Please only output the JSON structure."""

    chat_promp = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_message
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]

    try:
        completion = client.chat.completions.create(
            model=deployment,
            messages=chat_promp,
            max_tokens=4096,
            temperature=0,
            frequency_penalty=0,
            presence_penalty=0,
            stream=False
        )

        content = completion.choices[0].message.content.strip()
        
        if content.startswith("```") and content.endswith("```"):
            content = content[3:-3].strip()
        
        if content.startswith("json"):
            content = content[4:].strip()

        try:
            parsed = json.loads(content)
            parsed["person"] = int(df_note["MRN"][i])
            parsed["note_date"] = str(df_note["note_date"][i])
            parsed["note_id"] = int(df_note["note_id"][i])
            parsed["note_title"] = str(df_note["note_title"][i])
            parsed["GIRA_date"] = str(df_note["GIRA_date"][i])
            all_outputs.append(parsed)
        except json.JSONDecodeError:
            print(f"Warning: Could not parse JSON for index {i}. Content was:\n{content_clean}")
            all_outputs.append({"error": "Invalid JSON", "index": i, "raw": content_clean})

    except Exception as e:
        print(f"Error during inference at index {i}: {e}")
        all_outputs.append({"error": "Model error", "index": i, "exception": str(e)})

with open("GPT4o_baseline.json", "w", encoding="utf-8") as f:
    json.dump(all_outputs, f, ensure_ascii=False, indent=2)


In [ ]:
def safe_json_loads(raw_str):
    try:
        return json.loads(raw_str)
    except json.JSONDecodeError:
        try:
            fixed = raw_str.replace("\\\'", "'") 
            return json.loads(fixed)
        except Exception as e2:
            return None

with open('GPT4o_baseline.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

rows = []
for entry in data:
    parsed = None

    if 'error' in entry and 'raw' in entry:
        parsed = safe_json_loads(entry['raw'])
        index = entry.get("index")
        if parsed and parsed.get("referred") == True:
            entry = parsed
            person = int(df_note.loc[index, "MRN"])
            note_date = str(df_note.loc[index, "note_date"])
            GIRA_date = str(df_note.loc[index, "GIRA_date"])
            note_title = str(df_note.loc[index, "note_title"])
            note_id = int(df_note.loc[index, "note_id"])
        else:
            print(f"Failed to parse index: {index}")

            continue
            
    elif entry.get("referred") == True:
        parsed = entry
        person = parsed.get("person")
        note_date = parsed.get("note_date")
        GIRA_date = parsed.get("GIRA_date")
        note_title = parsed.get("note_title")
        note_id = parsed.get("note_id")
    else:
        continue

    specialists = parsed.get("specialists", [])
    for specialist in specialists:
        row = {
            "person": person,
            "GIRA_date": GIRA_date,
            "note_date": note_date,
            "note_id": note_id,
            "note_title": note_title,
            "specialist_type": specialist.get("specialist_type"),
            "referral_confirmation_citation": specialist.get("referral_confirmation_citation"),
            "specialist_type_citation": specialist.get("specialist_type_citation")
        }
        rows.append(row)
        
df = pd.DataFrame(rows)
